In [1]:
import pandas as pd

obj_value_df = pd.DataFrame(columns = ['obj_value_avg_1', 'obj_value_het_1', 'obj_value_avg_2', 'obj_value_het_2', 'obj_value_avg_3', 'obj_value_het_3'])
obj_value_better_df = pd.DataFrame(columns = ['obj_value_avg_1', 'obj_value_het_1', 'obj_value_avg_2', 'obj_value_het_2', 'obj_value_avg_3', 'obj_value_het_3'])
avg_work_day_sim = []
improved_weeks_sim = []

In [2]:
data_ori = pd.read_excel("ICU_avg_effect.xlsx", sheet_name = "Old Schedule")
data_ori

,weeknum,index,status,surgeon,weekday,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18
0,0,1,0,5,3,NaN,NaN,0,elective,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,2,0,5,3,NaN,NaN,1,urgent,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,3,0,1,3,NaN,NaN,2 or 3,emergent,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,4,0,1,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,5,0,5,4,NaN,NaN,smooth surgeon's workload across days,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5344,208,27,1,5,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5345,208,28,0,3,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5346,208,29,2,3,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5347,208,30,2,3,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
import multiprocessing
from ipynb.fs.full.simulation_worker import urgent_smooth
from ipynb.fs.full.simulation_worker import cal_obj1_urgent
from ipynb.fs.full.simulation_worker import mono_check
from ipynb.fs.full.simulation_worker import elective_smooth
from ipynb.fs.full.simulation_worker import cal_obj1
from ipynb.fs.full.simulation_worker import cal_obj2
from ipynb.fs.full.simulation_worker import additional_day
from ipynb.fs.full.simulation_worker import schedule_es
from ipynb.fs.full.simulation_worker import objective_cal
from ipynb.fs.full.simulation_worker import worker

In [ ]:
frac = 0.5 # fraction of the urgent cases to be considered as unknown in the first place

with multiprocessing.Pool(processes = 5) as pool:
    results = [
        pool.apply_async(
            worker, 
            args = (data_ori, obj_value_df, obj_value_better_df, avg_work_day_sim, improved_weeks_sim, frac)
        ) 
        for _ in range(5)
    ]
    output = [r.get() for r in results]
output

100%|██████████| 20/20 [42:52<00:00, 128.64s/it]


[(    obj_value_avg_1  obj_value_het_1  obj_value_avg_2  obj_value_het_2  \
  0          4221.945         4295.043        10290.411         8161.425   
  1          4231.045         4292.323        10312.591         8280.195   
  2          4234.685         4297.999        10321.463         8270.565   
  3          4209.205         4283.917        10259.359         8121.300   
  4          4218.305         4291.837        10281.539         8150.190   
  5          4229.225         4299.935        10308.155         8196.735   
  6          4259.255         4333.177        10381.349         8232.045   
  7          4247.425         4319.575        10352.515         8222.415   
  8          4262.895         4329.353        10390.221         8302.665   
  9          4278.365         4344.451        10427.927         8337.975   
  10         4270.175         4333.295        10407.965         8346.000   
  11         4212.845         4277.053        10268.231         8217.600   
  12        

In [5]:
# obj_value_df = pd.concat([output[0][0], output[1][0], output[2][0], output[3][0], output[4][0]], ignore_index = True)
# obj_value_df

In [ ]:
# objective value under rule better (do not change the weekly schedule if it is worse than the original schedule)
obj_value_better_df = pd.concat([output[0][1], output[1][1], output[2][1], output[3][1], output[4][1]], ignore_index = True)
obj_value_better_df

,obj_value_avg_1,obj_value_het_1,obj_value_avg_2,obj_value_het_2,obj_value_avg_3,obj_value_het_3
0,3915.275,4021.755,9542.945,7156.695,10033.430,7343.973
1,3916.185,4017.563,9545.163,7208.055,10035.762,7396.677
2,3911.635,4017.587,9534.073,7143.855,10024.102,7330.797
3,3928.015,4033.059,9573.997,7182.375,10066.078,7370.325
4,3902.535,4011.199,9511.893,7100.520,10000.782,7286.328
...,...,...,...,...,...,...
95,3912.545,4017.023,9536.291,7182.375,10026.434,7370.325
96,3931.655,4035.119,9582.869,7180.770,10075.406,7368.678
97,3920.735,4026.083,9556.253,7147.065,10047.422,7334.091
98,3948.035,4054.973,9622.793,7209.660,10117.382,7398.324


In [7]:
avg_work_day_sim = output[0][2] + output[1][2] + output[2][2] + output[3][2] + output[4][2]
avg_work_day_sim

[2.633047210300429,
 2.618271954674221,
 2.653486700215672,
 2.6327845382963493,
 2.6432664756446993,
 2.6402877697841727,
 2.627324749642346,
 2.6061036195883607,
 2.631200575125809,
 2.6216022889842634,
 2.6081370449678802,
 2.6539842067480257,
 2.6489208633093524,
 2.644396551724138,
 2.5825174825174826,
 2.6136363636363638,
 2.6153846153846154,
 2.636947218259629,
 2.6431146359048308,
 2.620443173695497,
 2.6421952957947257,
 2.6295503211991433,
 2.6051502145922747,
 2.6285919540229883,
 2.6551475881929445,
 2.6291814946619216,
 2.6221264367816093,
 2.5912305516265914,
 2.6464646464646466,
 2.608291636883488,
 2.6195965417867435,
 2.653179190751445,
 2.6379928315412187,
 2.6139105748757983,
 2.6446043165467628,
 2.640988372093023,
 2.593727726300784,
 2.6284697508896797,
 2.636039886039886,
 2.6140602582496415,
 2.619557458957887,
 2.661115133960898,
 2.655890804597701,
 2.645369705671213,
 2.617961511047755,
 2.6284896206156048,
 2.663761801016703,
 2.6523947750362846,
 2.64065855

In [8]:
improved_weeks_sim = output[0][3] + output[1][3] + output[2][3] + output[3][3] + output[4][3]
improved_weeks_sim

[83,
 86,
 91,
 91,
 92,
 90,
 84,
 89,
 78,
 79,
 82,
 94,
 83,
 78,
 91,
 85,
 77,
 85,
 84,
 88,
 93,
 89,
 84,
 84,
 93,
 91,
 89,
 85,
 81,
 80,
 80,
 84,
 99,
 89,
 83,
 82,
 81,
 92,
 82,
 83,
 89,
 88,
 84,
 90,
 95,
 82,
 79,
 91,
 77,
 85,
 94,
 87,
 87,
 82,
 86,
 77,
 87,
 83,
 84,
 88,
 78,
 83,
 86,
 94,
 99,
 87,
 86,
 92,
 83,
 85,
 89,
 97,
 85,
 85,
 91,
 84,
 90,
 79,
 83,
 90,
 87,
 79,
 97,
 87,
 85,
 96,
 84,
 83,
 87,
 85,
 82,
 86,
 81,
 79,
 96,
 85,
 85,
 89,
 83,
 87]

In [ ]:
# average_obj_value = pd.DataFrame({'obj_value_avg_1': obj_value_df['obj_value_avg_1'].sum()/100, 'obj_value_het_1': obj_value_df['obj_value_het_1'].sum()/100,
# 'obj_value_avg_2': obj_value_df['obj_value_avg_2'].sum()/100, 'obj_value_het_2': obj_value_df['obj_value_het_2'].sum()/100, 
# 'obj_value_avg_3': obj_value_df['obj_value_avg_3'].sum()/100, 'obj_value_het_3': obj_value_df['obj_value_het_3'].sum()/100}, index = [0])

# average_obj_value

,obj_value_avg_1,obj_value_het_1,obj_value_avg_2,obj_value_het_2,obj_value_avg_3,obj_value_het_3
0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
average_obj_value_better = pd.DataFrame({'obj_value_avg_1': obj_value_better_df['obj_value_avg_1'].sum()/100, 'obj_value_het_1': obj_value_better_df['obj_value_het_1'].sum()/100,
'obj_value_avg_2': obj_value_better_df['obj_value_avg_2'].sum()/100, 'obj_value_het_2': obj_value_better_df['obj_value_het_2'].sum()/100, 
'obj_value_avg_3': obj_value_better_df['obj_value_avg_3'].sum()/100, 'obj_value_het_3': obj_value_better_df['obj_value_het_3'].sum()/100}, index = [0])

average_obj_value_better

,obj_value_avg_1,obj_value_het_1,obj_value_avg_2,obj_value_het_2,obj_value_avg_3,obj_value_het_3
0,3925.1212,4030.21798,9566.94376,7178.15385,10058.66224,7365.99339


In [11]:
import numpy as np

avg_work_day_sim = np.array(avg_work_day_sim)
avg_work_day_result = np.mean(avg_work_day_sim)
avg_work_day_result

2.628656586626553

In [12]:
improved_weeks_sim = np.array(improved_weeks_sim)
improved_weeks_avg = np.average(improved_weeks_sim)
improved_weeks_avg

86.18

In [13]:
ori_obj_value_avg_1 = 0
ori_obj_value_avg_2 = 0
ori_obj_value_avg_3 = 0
ori_obj_value_het_1 = 0
ori_obj_value_het_2 = 0
ori_obj_value_het_3 = 0

for week in data_ori['weeknum'].unique():
    df_week = data_ori[data_ori['weeknum'] == week]
    for surgeon in df_week['surgeon'].unique():
        df_surgeon = df_week[df_week['surgeon'] == surgeon]
        _1, _2, _3, _4, _5, _6 = objective_cal(df_surgeon)
        
        ori_obj_value_avg_1 += _1
        ori_obj_value_avg_2 += _2
        ori_obj_value_avg_3 += _3
        ori_obj_value_het_1 += _4
        ori_obj_value_het_2 += _5
        ori_obj_value_het_3 += _6
    
ori_obj_value_df = pd.DataFrame({'ori_obj_value_avg_1': ori_obj_value_avg_1, 'ori_obj_value_het_1': ori_obj_value_het_1, 'ori_obj_value_avg_2': ori_obj_value_avg_2,'ori_obj_value_het_2': ori_obj_value_het_2, 'ori_obj_value_avg_3': ori_obj_value_avg_3, 'ori_obj_value_het_3': ori_obj_value_het_3}, index = [0])
ori_obj_value_df

,ori_obj_value_avg_1,ori_obj_value_het_1,ori_obj_value_avg_2,ori_obj_value_het_2,ori_obj_value_avg_3,ori_obj_value_het_3
0,4124.575,4240.775,10053.085,7594.86,10569.79,7793.604


In [14]:
import pandas as pd


# Create the DataFrame
res = {
    ' ': ['OR time', '(in hours)', 'Post-LOS', '(in days)', 'ICU Time', '(in days)'],
    'Effect': ['Avg', 'Het', 'Avg', 'Het', 'Avg', 'Het'],
    'dst_orig': ori_obj_value_df.values.flatten(),
    'dst': average_obj_value_better.values.flatten(),
    'delta': (ori_obj_value_df.values.flatten() - average_obj_value_better.values.flatten()),
    'rel_decr': (ori_obj_value_df.values.flatten() - average_obj_value_better.values.flatten()) / ori_obj_value_df.values.flatten(),
    'improve_week': pd.Series([improved_weeks_avg] * 6),
    'avg_work_day': pd.Series([avg_work_day_result] * 6)
}

res_df = pd.DataFrame(res)
res_df
res_df['dst_orig'] = res_df['dst_orig'].round(2)
res_df['dst'] = res_df['dst'].round(2)
res_df['delta'] = res_df['delta'].round(2)
res_df['rel_decr'] = (res_df['rel_decr'] * 100).round(2).astype(str) + '%'
res_df['avg_work_day'] = res_df['avg_work_day'].round(3)
res_df['improve_week'] = res_df['improve_week'].round(0)

# Save to Excel
res_df.to_excel(f'res/output_stochastic_random_{int(frac*100)}urgent_better.xlsx', index = False)